# Normalizing Flows — intuicja, matematyka i podstawy w PyTorch

> **Cel notebooka:** zbudować solidny fundament: gęstości prawdopodobieństwa, zmianę zmiennych, jakobian, estymację wiarygodności i kompromisy modeli generatywnych. Każda ważna zależność jest sprawdzona na małych, uruchamialnych przykładach.

Normalizing flow (NF) przekształca prosty rozkład bazowy, np. standardowy rozkład normalny, w złożony rozkład danych przez odwracalne mapy. Kluczowa zaleta: model potrafi zarówno **generować** próbki, jak i obliczać ich **dokładną log-gęstość**. Ceną jest wymóg odwracalności oraz efektywnego wyznaczenia wyznacznika jakobianu.

**Plan**
1. Gęstości, likelihood i zmiana zmiennych
2. Jakobian: znaczenie geometryczne i obliczenia
3. Kompozycja przepływów, log-determinant i stabilność
4. Minimalny przepływ w 1D: trening, próbkowanie i diagnostyka
5. Kiedy normalizing flows są dobrym wyborem


## 1. Co modelujemy?

Mamy obserwacje $x\in\mathbb R^D$ z nieznanego rozkładu $p_{data}(x)$. Model probabilistyczny przypisuje im gęstość $p_\theta(x)$. Trening maximum likelihood minimalizuje ujemną średnią log-wiarygodność:

$$\mathcal L_{NLL}(\theta)=-\frac1N\sum_{n=1}^N\log p_\theta(x_n).$$

Wysoka gęstość oznacza, że model uważa punkt za prawdopodobny. Gęstość nie jest jednak prawdopodobieństwem punktu (dla zmiennych ciągłych prawdopodobieństwo pojedynczego punktu wynosi zero); prawdopodobieństwo dotyczy obszaru i jest całką gęstości.

W przepływie wybieramy łatwy do próbkowania rozkład bazowy $z\sim p_Z$, zwykle $\mathcal N(0,I)$, i odwracalną transformację $x=f_\theta(z)$. Odwrotną mapą jest $z=f_\theta^{-1}(x)$. W przeciwieństwie do GAN-a, nie uczymy dyskryminatora; w przeciwieństwie do typowego VAE, likelihood obserwacji jest obliczany dokładnie (pomijając ewentualną dequantyzację dyskretnych danych).


## 2. Zmiana zmiennych — skąd bierze się wyznacznik?

W 1D transformacja $x=f(z)$ rozciąga mały odcinek $dz$ do $dx\approx |f'(z)|dz$. Zachowanie masy wymaga $p_X(x)dx=p_Z(z)dz$, więc $p_X(x)=p_Z(z)/|f'(z)|$. W wielu wymiarach lokalna skala objętości to wartość bezwzględna wyznacznika jakobianu.

$$\log p_X(x)=\log p_Z(f^{-1}(x))+\log\left|\det\frac{\partial f^{-1}(x)}{\partial x}\right|.$$

Równoważnie, dla generatywnej mapy $x=f(z)$:
$$\log p_X(f(z))=\log p_Z(z)-\log\left|\det\frac{\partial f(z)}{\partial z}\right|.$$

**Intuicja znaku:** mapa, która lokalnie rozciąga przestrzeń 10-krotnie, rozcieńcza tę samą masę 10-krotnie. W kierunku odwrotnym log-gęstość dostaje $-\log 10$. Dla kolejnych map $f_K\circ\dots\circ f_1$ log-wyznaczniki sumują się, bo wyznacznik iloczynu jakobianów jest iloczynem wyznaczników.


In [ ]:
import math
import torch

torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)

# x = a*z+b: gęstość normalna po afinicznym przekształceniu.
a, b = 2.0, -0.7
z = torch.tensor([0.0, 1.0, -1.0])
x = a * z + b
log_pz = -0.5 * (z.square() + math.log(2 * math.pi))
log_px_change = log_pz - math.log(abs(a))
log_px_direct = -0.5 * (((x-b)/a).square() + math.log(2*math.pi)) - math.log(abs(a))
print('x:', x)
print('log p(x) przez zmianę zmiennych:', log_px_change)
print('Kontrola względem znanego rozkładu:', torch.allclose(log_px_change, log_px_direct))

## 3. Jakobian i jego koszt

Dla $f:\mathbb R^D\to\mathbb R^D$ jakobian $J_{ij}=\partial f_i/\partial z_j$ opisuje lokalne rozciąganie i ścinanie. Zmiana zmiennych wymaga $\log|\det J|$. Naiwne wyznaczenie macierzy i jej wyznacznika kosztuje zwykle $O(D^3)$, co jest niepraktyczne dla obrazu o tysiącach wymiarów.

Architektura flow jest więc konstruowana tak, by log-wyznacznik był tani: transformacje trójkątne, diagonalne, permutacje lub specjalnie parametryzowane macierze. Poniższy przykład pokazuje obliczenie autodiff dla małego wymiaru oraz zgodność z analitycznym wynikiem.


In [ ]:
def f(v):
    # Odwracalna mapa nieliniowa w 2D: (u,v) -> (u, v + 0.4 sin(u))
    return torch.stack([v[0], v[1] + 0.4 * torch.sin(v[0])])

point = torch.tensor([0.3, -0.2], requires_grad=True)
J = torch.autograd.functional.jacobian(f, point)
sign, logabsdet = torch.linalg.slogdet(J)
print('Jakobian:\n', J.detach())
print('det J =', torch.det(J).item(), '| log|det J| =', logabsdet.item())
print('Oczekiwany log|det J| = 0, bo mapa ma diagonalę (1, 1)')

## 4. Przepływ jako łańcuch odwracalnych warstw

Niech $h_0=z$ i $h_k=f_k(h_{k-1})$. Wtedy
$$\log p_X(x)=\log p_Z(h_0)-\sum_{k=1}^K\log|\det J_{f_k}(h_{k-1})|.$$

Każda warstwa musi mieć odwrotność (analityczną lub wydajnie obliczalną) i tani log-wyznacznik. Transformacje nieliniowe zwiększają ekspresywność; same permutacje czy transformacje liniowe nie wystarczą, by modelować złożone kształty. Wysokość/głębokość przepływu zwiększa możliwości, ale może zwiększyć koszt pamięci, obliczeń i trudność optymalizacji.

W praktyce obserwowane obrazy są dyskretne. Podanie ciągłemu modelowi liczb całkowitych bezpośrednio nie opisuje poprawnie prawdopodobieństwa mas punktów. Częsty zabieg to dequantyzacja: dodać szum $u\sim U([0,1)^D)$ do pikseli i optymalizować dolne ograniczenie log-likelihood dyskretnego obrazu. W demonstracjach 2D unikamy tego problemu.


## 5. Minimalny 1D flow: afiniczna mapa

Uczymy $x=\exp(s)z+t$, gdzie $z\sim\mathcal N(0,1)$. Wtedy
$$\log p_X(x)=\log p_Z((x-t)e^{-s})-s.$$

Ten model ma tylko dwa stopnie swobody, więc dopasuje przesunięcie i skalę, ale nie wielomodalny kształt. Jest użyteczny do sprawdzenia rachunku likelihood i całego cyklu optymalizacji.


In [ ]:
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(11)
# Dane o znanym, niegaussowskim kształcie: mieszanina dwóch klastrów.
n = 6000
component = torch.randint(0, 2, (n,), device=device)
data = torch.randn(n, device=device) * 0.35 + torch.where(component == 0, -2.0, 1.5)

class Affine1D(nn.Module):
    def __init__(self):
        super().__init__(); self.log_scale = nn.Parameter(torch.zeros(())); self.shift = nn.Parameter(torch.zeros(()))
    def log_prob(self, x):
        z = (x - self.shift) * torch.exp(-self.log_scale)
        return -0.5 * (z.square() + math.log(2 * math.pi)) - self.log_scale
    def sample(self, n):
        return torch.randn(n, device=self.shift.device) * torch.exp(self.log_scale) + self.shift

model = Affine1D().to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.04)
for step in range(800):
    batch = data[torch.randint(n, (256,), device=device)]
    loss = -model.log_prob(batch).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print('NLL treningowy:', round(loss.item(), 3), '| skala:', model.log_scale.exp().item(), '| przesunięcie:', model.shift.item())

grid = torch.linspace(-4, 4, 400, device=device)
with torch.no_grad():
    density = model.log_prob(grid).exp().cpu()
plt.figure(figsize=(8, 3)); plt.hist(data.cpu(), bins=70, density=True, alpha=.45, label='dane')
plt.plot(grid.cpu(), density, label='dopasowany 1D flow'); plt.legend(); plt.title('Jedna Gaussowska baza nie odtworzy dwóch mód'); plt.show()

## 6. Jak diagnozować flow?

- **NLL / bits per dimension:** porównuj te same dane, preprocessing i konwencję (dla obrazów także dequantyzację). Niższy NLL oznacza lepszą przypisaną gęstość na zbiorze ewaluacyjnym.
- **Próbki:** sprawdzają jakość wizualną, ale nie zastępują likelihood. Wysoka gęstość nie zawsze oznacza semantycznie atrakcyjny obraz.
- **Odwracalność numeryczna:** sprawdź, czy $f^{-1}(f(z))\approx z$ i czy sumy log-determinantów w przód i wstecz znoszą się.
- **Stabilność:** monitoruj skalowania, NaN/Inf i gradienty. Ograniczanie skali logarytmicznej bywa konieczne.
- **Train vs validation NLL:** duża luka sugeruje przeuczenie; walidację trzymaj poza optymalizacją.

**Zalety:** jawna i dokładna gęstość, proste próbkowanie z bazy, odwracalna reprezentacja. **Ograniczenia:** koszt na wymiar, kompromisy architektoniczne dla jakobianu, czasem gorsza jakość percepcyjna od GAN/diffusion, oraz znane zjawisko, że likelihood nie zawsze zgadza się z intuicyjną semantyczną jakością.

## Podsumowanie

Sednem normalizing flow jest równanie zmiany zmiennych. Uczymy odwracalną mapę między prostym rozkładem a danymi; log-gęstość obserwacji to log-gęstość punktu w bazie plus korekta objętości. Konstrukcja wydajnych flow polega na znalezieniu warstw, które są jednocześnie ekspresywne, odwracalne i mają tani log-wyznacznik. Następny notebook rozwija tę ideę w RealNVP.
